# ObjectOracle — Model Evaluation Notebook
Evaluate trained YOLOv8 and visualize detection results.

In [ ]:
import sys
sys.path.insert(0, '..')

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from pathlib import Path
from ultralytics import YOLO

plt.style.use('dark_background')
print('Ready')

## 1. Load Model

In [ ]:
MODEL_PATH = '../models/yolov8n.pt'  # change to your custom model
model = YOLO(MODEL_PATH)
print(f'Model: {MODEL_PATH}')
print(f'Classes ({len(model.names)}): {list(model.names.values())[:10]}...')

## 2. Detect on a Sample Image

In [ ]:
IMAGE_PATH = 'sample.jpg'  # provide a test image

results = model.predict(IMAGE_PATH, conf=0.4, verbose=False)
r = results[0]

img = Image.open(IMAGE_PATH).convert('RGB')
fig, ax = plt.subplots(1, 1, figsize=(12, 8))
ax.imshow(img)

COLORS = plt.cm.Set1.colors
for box in r.boxes:
    x1, y1, x2, y2 = box.xyxy[0].tolist()
    cls_id = int(box.cls)
    label  = model.names[cls_id]
    conf   = float(box.conf)
    color  = COLORS[cls_id % len(COLORS)]
    rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor=color, facecolor='none')
    ax.add_patch(rect)
    ax.text(x1, y1-5, f'{label} {conf:.2f}', color=color, fontsize=10, fontweight='bold')

ax.axis('off')
ax.set_title(f'{len(r.boxes)} objects detected', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Run Validation on Dataset

In [ ]:
DATASET_YAML = '../data/dataset.yaml'
metrics = model.val(data=DATASET_YAML, verbose=False)

print(f'mAP50:     {metrics.box.map50:.4f}')
print(f'mAP50-95:  {metrics.box.map:.4f}')
print(f'Precision: {metrics.box.mp:.4f}')
print(f'Recall:    {metrics.box.mr:.4f}')

## 4. Per-class mAP Bar Chart

In [ ]:
import pandas as pd

class_names = list(model.names.values())
map50_per_class = metrics.box.ap50.tolist()

df = pd.DataFrame({'class': class_names[:len(map50_per_class)], 'mAP50': map50_per_class})
df = df.sort_values('mAP50', ascending=True)

fig, ax = plt.subplots(figsize=(10, max(4, len(df) * 0.4)))
bars = ax.barh(df['class'], df['mAP50'], color='#7c6df5', edgecolor='none')
ax.set_xlabel('mAP50', fontsize=12)
ax.set_title('Per-class mAP50', fontsize=14, fontweight='bold')
ax.axvline(metrics.box.map50, color='#22d3a5', linestyle='--', linewidth=1.5, label=f'Mean: {metrics.box.map50:.3f}')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Confidence Distribution

In [ ]:
from glob import glob

image_paths = glob('../data/dataset/images/val/*.jpg')[:200]
all_confs = []

for path in image_paths:
    res = model.predict(path, conf=0.1, verbose=False)
    for box in res[0].boxes:
        all_confs.append(float(box.conf))

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(all_confs, bins=40, color='#7c6df5', edgecolor='#0f1117', alpha=0.85)
ax.axvline(0.4, color='#f59e0b', linestyle='--', label='Threshold=0.40')
ax.set_xlabel('Confidence score', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Detection confidence distribution', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()
print(f'Total detections (conf≥0.1): {len(all_confs)}')
print(f'Detections kept (conf≥0.40): {sum(1 for c in all_confs if c >= 0.4)}')

## 6. Inference Speed Benchmark

In [ ]:
import time

test_image = Image.new('RGB', (640, 640), color=(100, 150, 200))
N = 50
times = []

# Warm-up
model.predict(test_image, conf=0.4, verbose=False)

for _ in range(N):
    t0 = time.perf_counter()
    model.predict(test_image, conf=0.4, verbose=False)
    times.append((time.perf_counter() - t0) * 1000)

print(f'Inference over {N} runs:')
print(f'  Mean:   {np.mean(times):.1f} ms')
print(f'  Median: {np.median(times):.1f} ms')
print(f'  Min:    {np.min(times):.1f} ms')
print(f'  Max:    {np.max(times):.1f} ms')
print(f'  FPS:    {1000/np.mean(times):.1f}')

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(times, color='#7c6df5', linewidth=1.5)
ax.axhline(np.mean(times), color='#f59e0b', linestyle='--', label=f'Mean {np.mean(times):.1f}ms')
ax.set_xlabel('Run', fontsize=12)
ax.set_ylabel('ms', fontsize=12)
ax.set_title('Inference latency per run', fontsize=14)
ax.legend()
plt.tight_layout()
plt.show()